In [ ]:
import pandas as pd
import joblib

In [ ]:
model = joblib.load("../medicine_price_model.pkl")
encoders = joblib.load("../label_encoders.pkl")
original_df = pd.read_csv("../data/data.csv")

In [ ]:

df = pd.read_csv("../data/data.csv")


# Apply the same preprocessing you used during training

In [ ]:
columns_to_drop = [
    "product_id",
    "active_ingredients",
    "manufacturer_raw",
    "packaging_raw"
]
print(df.columns.tolist())
df = df.drop(columns=columns_to_drop, errors="ignore")

df["pack_size"] = df["pack_size"].fillna(df["pack_size"].median())
df["pack_unit"] = df["pack_unit"].fillna(df["pack_unit"].mode()[0])
df["primary_strength"] = df["primary_strength"].fillna("Unknown")

In [ ]:
original_df = df.copy()

In [ ]:
categorical_columns = [
    "brand_name",
    "manufacturer",
    "dosage_form",
    "pack_unit",
    "primary_ingredient",
    "primary_strength",
    "therapeutic_class"
]

for col in categorical_columns:
    df[col] = encoders[col].transform(df[col].astype(str))

In [ ]:
X = df.drop("price_inr", axis=1)

In [ ]:
predicted_prices = model.predict(X)

original_df["predicted_price"] = predicted_prices



In [ ]:
medicine = input("Enter the medicine name: ")

medicine_row = original_df[
    original_df["brand_name"].str.contains(
        medicine,
        case=False,
        na=False
    )
]

In [ ]:
if medicine_row.empty:
    print("Medicine not found.")
else:
    salt = medicine_row.iloc[0]["primary_ingredient"]

    print("Salt found:", salt)

In [ ]:
same_salt = original_df[
    original_df["primary_ingredient"].str.lower() == salt.lower()
]

In [ ]:
if same_salt.empty:
    print("No medicine found with this salt.")
else:
    cheapest = same_salt.loc[
        same_salt["predicted_price"].idxmin()
    ]

    print("Cheapest Medicine")
    print("------------------")
    print("Brand:", cheapest["brand_name"])
    print("Manufacturer:", cheapest["manufacturer"])
    print("Salt:", cheapest["primary_ingredient"])
    print("Predicted Price: ₹", round(cheapest["predicted_price"], 2))